In [ ]:
# %% [1. ตรวจสอบสภาพแวดล้อมฮาร์ดแวร์]
import torch

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"รันโมเดลบนฮาร์ดแวร์: {device}")

# %% [2. นำเข้าไลบรารีและตั้งค่าพารามิเตอร์หลัก]
import numpy as np
from scipy.io import loadmat
import torch.nn as nn
import random
import gc
import os
from datetime import datetime as dt
from torch.utils.data import DataLoader, TensorDataset
import evaluate as ev
import visualizer as vis

ai_model = 'vit'
scenario = 'O1'
antennas = 64
epochs = 100
save_path = './best_models/'
os.makedirs(save_path, exist_ok=True)

# กำหนดสเปกตรัมความถี่และช่วง SNR สำหรับ Auto ML 
frequencies = [28, 60, 140]
snr_list = [0, 5, 10, 15, 20]

def apply_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    print(f"Global environment locked with seed: {seed}")

# ==========================================
# สถาปัตยกรรม Vision Transformer (ViT)
# ==========================================
class BeamPredictionViT(nn.Module):
    def __init__(self, in_channels=2, img_size=(64, 32), patch_size=8, 
                 d_model=64, num_heads=4, num_layers=2, n_beams=64, dropout_rate=0.2):
        super(BeamPredictionViT, self).__init__()
        
        # คำนวณจำนวนชิ้นส่วน (Patches)
        self.num_patches = (img_size[0] // patch_size) * (img_size[1] // patch_size)
        
        # Patch Extraction + Linear Embedding
        self.patch_embed = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        
        # CLS Token และ Learnable Positional Encoding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, d_model))
        self.pos_drop = nn.Dropout(p=dropout_rate)
        
        # Transformer Encoder Block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, 
            dim_feedforward=d_model*4, dropout=dropout_rate, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output Classifier
        self.fc = nn.Linear(d_model, n_beams)

    def forward(self, x):
        B = x.shape[0]
        
        # หั่นภาพเป็น Patch และบีบอัดมิติ
        x = self.patch_embed(x)        
        x = x.flatten(2).transpose(1, 2)
        
        # นำ CLS Token มาต่อคิวไว้ข้างหน้าสุด
        cls_tokens = self.cls_token.expand(B, -1, -1)  
        x = torch.cat((cls_tokens, x), dim=1)          
        
        # บวก Positional Encoding
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # ส่งเข้า Multi-Head Attention
        x = self.transformer_encoder(x) 
        
        # ดึง Index 0 (CLS Token) ออกมาฟันธง
        cls_out = x[:, 0, :] 
        
        # ทำนายผล Beam
        out = self.fc(cls_out) 
        return out

# %% [3. เริ่มต้นระบบประมวลผลลูป Auto ML ทั่วทั้งตารางกริดทดลอง]
for frequency in frequencies:
    for snr in snr_list:
        print(f"\n" + "="*70)
        print(f"--- Start configuration: {ai_model.upper()} | Frequency {frequency} GHz | SNR {snr} dB ---")
        print("="*70)
        
        # 1. โหลดข้อมูล CSI
        path = f'../DeepMIMO/DeepMIMO/DeepMIMO_dataset/SNR{snr}dB_{scenario}_{frequency}_Ant{antennas}/'
        
        try:
            d1 = loadmat(path+'channel1.mat')['a']
            d2 = loadmat(path+'channel2.mat')['b']
            d3 = loadmat(path+'channel3.mat')['c']
        except FileNotFoundError:
            print(f"[ERROR] ไม่พบไฟล์ที่ {path} ข้ามไปลูปถัดไป...")
            continue

        data = np.concatenate((d1, d2, d3), axis=2).transpose(2, 0, 1)

        # ล้างหน่วยความจำทันทีที่รวมไฟล์เสร็จ
        del d1, d2, d3 
        gc.collect()

        # จัดมิติข้อมูลเป็นรูปภาพ 2D: (Samples, 2, 64, 32) สำหรับ ViT
        d_r = data.real.reshape(-1, 1, 64, 32)
        d_i = data.imag.reshape(-1, 1, 64, 32)
        X = np.concatenate((d_r, d_i), axis=1).astype(np.float32)

        del data, d_r, d_i
        gc.collect()

        # Normalization
        mean = np.mean(X, axis=0)
        std = np.std(X, axis=0)
        X = (X - mean) / (std + 1e-8)

        print(f"Data ready! Input shape (ViT format): {X.shape}")
        
        # โหลดไฟล์ข้อมูลฉลาก และ Spectral Efficiency
        y = loadmat(path+'DLCB_output.mat')['onehot_label'].astype(np.float32)
        se_data = loadmat(path+'rate_ave.mat')['DL_output'].astype(np.float32)

        # 2. กระบวนการแบ่งส่วนข้อมูล (70% train, 30% test)
        split_idx = int(len(X) * 0.7)

        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        se_test = se_data[split_idx:]

        train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
        test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

        # 3. ล็อคระบบความสุ่มทุกลูปเพื่อให้โมเดลแข่งกันอย่างยุติธรรม
        apply_global_seed(42)

        # สร้างโมเดล ViT (รีเซ็ตน้ำหนักใหม่ทุกลูป)
        model = BeamPredictionViT(
            in_channels=2, 
            img_size=(64, 32), 
            patch_size=8, 
            d_model=64, 
            num_heads=4, 
            num_layers=2, 
            n_beams=64, 
            dropout_rate=0.2
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
        criterion = nn.CrossEntropyLoss()

        params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'Total Trainable Params: {params}')

        # 4. ขั้นตอนการเทรน
        best_loss = float('inf')
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            epoch_loss = 0.0
            model.train()
            
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, torch.max(labels, 1)[1])
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            val_loss = 0.0
            model.eval()
            with torch.no_grad():
                for inputs, labels in test_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, torch.max(labels, 1)[1])
                    val_loss += loss.item()
            
            avg_val_loss = val_loss / len(test_loader)
            val_losses.append(avg_val_loss)
                
            scheduler.step(avg_val_loss)
                
            if (epoch + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                torch.save(model.state_dict(), save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth')

        print(f"--> Finished. Best Validation Loss: {best_loss:.4f}")

        # 5. ประเมินผลและเรนเดอร์กราฟ
        model.load_state_dict(torch.load(save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth'))
        model.eval()

        eval_datetime = dt.now().strftime("%Y-%m-%d %H:%M:%S")
        ds_config = {'snr': snr, 'scenario': scenario, 'frequency': frequency, 'antennas': antennas}  

        mimo_results = ev.evaluate_performance(
            model, test_loader, device, criterion, ai_model, ds_config, eval_datetime
        )
        
        # วาดกราฟและบันทึกภาพ
        vis.plot_training_loss(train_losses, val_losses, mimo_results, ds_config, eval_datetime)
        vis.plot_confusion_matrix(mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)
        vis.plot_beam_tracking(mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)
        
        # 6. คัดแยกและพล็อตกราฟประสิทธิภาพช่องสัญญาณ SE
        preds_array = np.array(mimo_results['all_preds'])
        actuals_array = np.array(mimo_results['all_actuals'])
        
        num_users = len(preds_array)
        user_indices = np.arange(num_users)
        
        predicted_se = se_test[user_indices, preds_array]
        optimal_se = se_test[user_indices, actuals_array]

        plot_limit = 200
        vis.plot_se_tracking(
            user_indices=user_indices[:plot_limit], 
            optimal_se=optimal_se[:plot_limit],
            predicted_se=predicted_se[:plot_limit], 
            model_name=ai_model, 
            ds_config=ds_config, 
            eval_datetime=eval_datetime
        )

        vis.save_se_summary_to_csv(
            mimo_results=mimo_results, 
            se_test=se_test, 
            model_name=ai_model, 
            ds_config=ds_config, 
            eval_datetime=eval_datetime
        )
        
        # 7. คืนพื้นที่ RAM และ VRAM เต็มรูปแบบสำหรับลูปรอบถัดไป
        del X, y, se_data
        del X_train, X_test, y_train, y_test, se_test
        del train_ds, test_ds, train_loader, test_loader, model, optimizer, scheduler, criterion
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

print("\n" + "="*50)
print(f"=== ALL CONFIGURATIONS FOR {ai_model.upper()} COMPLETE ===")
print("="*50)